# Vocal-Guard — Evaluasi Resmi pada ASVspoof 2019 LA Eval Set & Kuantisasi INT8
**Tujuan:**
1. Mengukur performa generalisasi model **M1 (LCNN)** pada 71,237 utterance **Eval Set** (serangan unseen A07–A19).
2. Mengukur degradasi performa (drop EER) model **FP32 vs ONNX INT8** pada Dev Set dan Eval Set.
3. Menghasilkan tabel metrik resmi untuk publikasi & esai teknis.

In [ ]:
!pip install -q soundfile scikit-learn pandas scipy tqdm matplotlib onnx onnxruntime onnxscript

In [ ]:
import os, sys, glob, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.fftpack import dct
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import onnx
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print('🔥 PyTorch Device:', device.type.upper())
if torch.cuda.is_available():
    print('   GPU Model:', torch.cuda.get_device_name(0))
print('=' * 60)

In [ ]:
# Deteksi lokasi protokol secara fleksibel
kaggle_protocols = list(Path('/kaggle/input').glob('**/ASVspoof2019.LA.cm.train.trn.txt'))
if not kaggle_protocols:
    raise FileNotFoundError('Protokol ASVspoof2019 LA tidak ditemukan di /kaggle/input!')

PROTOCOL_DIR = kaggle_protocols[0].parent
LA_ROOT = PROTOCOL_DIR.parent

def find_flac_dir(root_dir, split_name):
    candidates = list(root_dir.glob(f'**/*{split_name}*/**/flac'))
    if not candidates:
        candidates = list(Path('/kaggle/input').glob(f'**/*{split_name}*/**/flac'))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f'Folder FLAC untuk split {split_name} tidak ditemukan!')

dev_audio_dir = find_flac_dir(LA_ROOT, 'dev')
eval_audio_dir = find_flac_dir(LA_ROOT, 'eval')

def load_protocol(protocol_file):
    df = pd.read_csv(
        protocol_file,
        sep=r'\s+',
        header=None,
        names=['speaker_id', 'audio_id', 'dash', 'system_id', 'label'],
        dtype=str
    )
    df['label'] = df['label'].str.lower()
    return df

# Cari protokol eval
eval_proto_candidates = list(Path('/kaggle/input').glob('**/ASVspoof2019.LA.cm.eval.trl.txt'))
if not eval_proto_candidates:
    eval_proto_candidates = list(PROTOCOL_DIR.glob('*eval*.txt'))
if not eval_proto_candidates:
    raise FileNotFoundError('Protokol Eval tidak ditemukan!')

dev_df = load_protocol(PROTOCOL_DIR / 'ASVspoof2019.LA.cm.dev.trl.txt')
eval_df = load_protocol(eval_proto_candidates[0])

print('=' * 60)
print('✅ Dataset & Protokol Berhasil Dimuat:')
print('   Dev samples: ', len(dev_df), '(Bonafide:', (dev_df['label'] == 'bonafide').sum(), 'Spoof:', (dev_df['label'] == 'spoof').sum(), ')')
print('   Eval samples:', len(eval_df), '(Bonafide:', (eval_df['label'] == 'bonafide').sum(), 'Spoof:', (eval_df['label'] == 'spoof').sum(), ')')
print('   Dev FLAC Dir: ', dev_audio_dir)
print('   Eval FLAC Dir:', eval_audio_dir)
print('=' * 60)

In [ ]:
def get_linear_filterbank(n_filters=20, n_fft=512, sr=16000, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sr / 2.0
    freq_points = np.linspace(low_freq, high_freq, n_filters + 2)
    bin_points = np.floor((n_fft + 1) * freq_points / sr).astype(int)
    n_bins = n_fft // 2 + 1
    filterbank = np.zeros((n_filters, n_bins))
    for i in range(n_filters):
        left, center, right = bin_points[i], bin_points[i + 1], bin_points[i + 2]
        if center > left:
            for b in range(left, center):
                if b < n_bins:
                    filterbank[i, b] = (b - left) / (center - left)
        if right > center:
            for b in range(center, right):
                if b < n_bins:
                    filterbank[i, b] = (right - b) / (right - center)
    return filterbank

def compute_deltas(feats, N=2):
    T, D = feats.shape
    deltas = np.zeros_like(feats)
    denom = 2 * sum(n**2 for n in range(1, N + 1))
    for t in range(T):
        num = np.zeros(D)
        for n in range(1, N + 1):
            t_plus = min(T - 1, t + n)
            t_minus = max(0, t - n)
            num += n * (feats[t_plus] - feats[t_minus])
        deltas[t] = num / denom
    return deltas

def extract_lfcc(y, sr=16000, window_ms=20.0, hop_ms=10.0, n_fft=512, num_filters=20, num_coeffs=20):
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])
    frame_len = int(sr * window_ms / 1000.0)
    hop_len = int(sr * hop_ms / 1000.0)
    if len(y) < frame_len:
        y = np.pad(y, (0, frame_len - len(y)))
    num_frames = 1 + (len(y) - frame_len) // hop_len
    shape = (num_frames, frame_len)
    strides = (y.strides[0] * hop_len, y.strides[0])
    frames = np.lib.stride_tricks.as_strided(y, shape=shape, strides=strides).copy()
    frames *= np.hanning(frame_len)
    mag_spec = np.abs(np.fft.rfft(frames, n=n_fft, axis=-1))
    fbank = get_linear_filterbank(n_filters=num_filters, n_fft=n_fft, sr=sr)
    filter_energies = np.dot(mag_spec, fbank.T)
    log_energies = np.log(filter_energies + 1e-12)
    static_lfcc = dct(log_energies, type=2, axis=-1, norm='ortho')[:, :num_coeffs]
    delta = compute_deltas(static_lfcc, N=2)
    delta2 = compute_deltas(delta, N=2)
    return np.hstack([static_lfcc, delta, delta2])

def pad_or_truncate(audio, target_length=64000):
    length = len(audio)
    if length == target_length:
        return audio
    if length > target_length:
        return audio[:target_length]
    repeat_factor = int(np.ceil(target_length / length))
    repeated = np.tile(audio, repeat_factor)
    return repeated[:target_length]

class ASVspoof2019Dataset(Dataset):
    def __init__(self, df, audio_dir, target_length=64000, sr=16000):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.target_length = target_length
        self.sr = sr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = self.audio_dir / f'{row["audio_id"]}.flac'
        try:
            audio, sr = sf.read(str(file_path), dtype='float32')
        except Exception:
            audio = np.zeros(self.target_length, dtype=np.float32)
        audio = pad_or_truncate(audio, target_length=self.target_length)
        lfcc = extract_lfcc(audio, sr=self.sr)  # (399, 60)
        feat = torch.from_numpy(lfcc.T).float().unsqueeze(0)  # (1, 60, T)
        label = 0 if row['label'] == 'bonafide' else 1
        return feat, label, row['audio_id'], row['system_id']

In [ ]:
class MaxFeatureMap2D(nn.Module):
    def forward(self, x):
        c = x.size(1) // 2
        return torch.max(x[:, :c, :, :], x[:, c:, :, :])

class MaxFeatureMap1D(nn.Module):
    def forward(self, x):
        d = x.size(1) // 2
        return torch.max(x[:, :d], x[:, d:])

class ConvMFM2D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 2 * out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(2 * out_channels)
        self.mfm = MaxFeatureMap2D()

    def forward(self, x):
        return self.mfm(self.bn(self.conv(x)))

class LinearMFM(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 2 * out_features)
        self.mfm = MaxFeatureMap1D()

    def forward(self, x):
        return self.mfm(self.linear(x))

class LCNN(nn.Module):
    def __init__(self, num_classes=2, in_channels=1, dropout_p=0.5):
        super().__init__()
        self.layer1 = nn.Sequential(
            ConvMFM2D(in_channels, 32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            ConvMFM2D(32, 32, kernel_size=3, stride=1, padding=1),
            ConvMFM2D(32, 48, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer3 = nn.Sequential(
            ConvMFM2D(48, 48, kernel_size=3, stride=1, padding=1),
            ConvMFM2D(48, 64, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer4 = nn.Sequential(
            ConvMFM2D(64, 64, kernel_size=3, stride=1, padding=1),
            ConvMFM2D(64, 64, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((3, 4))
        self.flat_dim = 64 * 3 * 4  # 768
        self.fc1 = LinearMFM(self.flat_dim, 160)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(160, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        emb = self.fc1(x)
        emb = self.dropout(emb)
        logits = self.fc2(emb)
        return logits

In [ ]:
# Cari checkpoint model terbaik dari run sebelumnya atau download lokal
checkpoint_candidates = list(Path('/kaggle/input').glob('**/lcnn_best_model.pt'))
if checkpoint_candidates:
    ckpt_path = str(checkpoint_candidates[0])
elif Path('lcnn_best_model.pt').exists():
    ckpt_path = 'lcnn_best_model.pt'
else:
    # URL checkpoint alternatif jika diperlukan
    raise FileNotFoundError('lcnn_best_model.pt tidak ditemukan!')

print('Loading PyTorch LCNN Model from:', ckpt_path)
model = LCNN(num_classes=2).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()
print('✅ Model LCNN loaded successfully!')

# Export ONNX & Dynamic INT8
model_cpu = LCNN(num_classes=2)
model_cpu.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
model_cpu.eval()
dummy_input = torch.randn(1, 1, 60, 399)

onnx_fp32_path = 'lcnn_m1_4s.onnx'
onnx_int8_path = 'lcnn_m1_4s_int8.onnx'

torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_fp32_path,
    export_params=True,
    opset_version=17,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    dynamo=False,
)
print('✅ Exported ONNX FP32:', onnx_fp32_path)

quantize_dynamic(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_path,
    weight_type=QuantType.QInt8,
)
print('✅ Quantized Dynamic INT8 ONNX created:', onnx_int8_path)

In [ ]:
def compute_eer(bonafide_scores, spoof_scores):
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    scores = np.concatenate([bonafide_scores, spoof_scores])
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer), float(thresholds[idx])

def evaluate_dataset(model, dataloader, device, desc='Evaluating'):
    model.eval()
    all_scores, all_labels, all_attacks, all_ids = [], [], [], []
    with torch.no_grad():
        for feats, labels, audio_ids, attacks in tqdm(dataloader, desc=desc):
            feats = feats.to(device)
            logits = model(feats)
            scores = (logits[:, 0] - logits[:, 1]).cpu().numpy()
            all_scores.extend(scores)
            all_labels.extend(labels.numpy())
            all_attacks.extend(attacks)
            all_ids.extend(audio_ids)

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    all_attacks = np.array(all_attacks)
    bona_scores = all_scores[all_labels == 0]
    spoof_scores = all_scores[all_labels == 1]
    eer, thresh = compute_eer(bona_scores, spoof_scores)
    return eer, thresh, bona_scores, spoof_scores, all_scores, all_labels, all_attacks, all_ids

BATCH_SIZE = 64
dev_dataset = ASVspoof2019Dataset(dev_df, dev_audio_dir)
eval_dataset = ASVspoof2019Dataset(eval_df, eval_audio_dir)

dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
print('=' * 70)
print('🚀 MEMULAI EVALUASI RESMI M1 (LCNN FP32) PADA ASVSPOOF 2019 LA')
print('=' * 70)

# 1. Evaluasi Dev Set (24,844 utterances)
dev_eer, dev_thresh, dev_bona, dev_spoof, _, _, _, _ = evaluate_dataset(model, dev_loader, device, desc='Dev Set (24,844)')
print(f'✅ Dev Set EER:   {dev_eer * 100:.2f}% | Threshold: {dev_thresh:.4f}')

# 2. Evaluasi Eval Set (71,237 utterances — Unseen Attacks A07-A19)
eval_eer, eval_thresh, eval_bona, eval_spoof, eval_scores, eval_labels, eval_attacks, eval_ids = evaluate_dataset(model, eval_loader, device, desc='Eval Set (71,237)')
print(f'🏆 EVAL SET EER:  {eval_eer * 100:.2f}% | Threshold: {eval_thresh:.4f}')
print('=' * 70)

# Breakdown per attack type (A07 - A19)
attack_breakdown = []
unique_attacks = sorted(list(set(eval_attacks)))
for atk in unique_attacks:
    if atk in ['bonafide', '-']:
        continue
    atk_scores = eval_scores[eval_attacks == atk]
    atk_eer, _ = compute_eer(eval_bona, atk_scores)
    attack_breakdown.append({
        'Attack': atk,
        'Samples': len(atk_scores),
        'EER (%)': round(atk_eer * 100, 2),
        'Mean Score': round(float(atk_scores.mean()), 4)
    })

atk_df = pd.DataFrame(attack_breakdown)
atk_df.to_csv('eval_attack_breakdown.csv', index=False)
print('
📊 Rincian Performa per Serangan (Unseen A07-A19):')
print(atk_df.to_string(index=False))

In [ ]:
print('
' + '=' * 70)
print('⚡ EVALUASI DEGRADASI KUANTISASI (FP32 vs INT8 ONNX)')
print('=' * 70)

sess_fp32 = ort.InferenceSession('lcnn_m1_4s.onnx', providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession('lcnn_m1_4s_int8.onnx', providers=['CPUExecutionProvider'])
in_name_fp32 = sess_fp32.get_inputs()[0].name
in_name_int8 = sess_int8.get_inputs()[0].name

# Uji pada 2,000 sampel Dev set untuk melihat akurasi drop numerik INT8
sub_bona_fp32, sub_spoof_fp32 = [], []
sub_bona_int8, sub_spoof_int8 = [], []

subset_size = min(2000, len(dev_dataset))
indices = np.random.RandomState(42).choice(len(dev_dataset), size=subset_size, replace=False)

for idx in tqdm(indices, desc='CPU INT8 vs FP32 Parity'):
    feat, label, _, _ = dev_dataset[idx]
    feat_np = feat.numpy()  # (1, 60, 399)
    out_fp32 = sess_fp32.run(None, {in_name_fp32: feat_np})[0]
    out_int8 = sess_int8.run(None, {in_name_int8: feat_np})[0]
    score_fp32 = out_fp32[0, 0] - out_fp32[0, 1]
    score_int8 = out_int8[0, 0] - out_int8[0, 1]
    if label == 0:
        sub_bona_fp32.append(score_fp32)
        sub_bona_int8.append(score_int8)
    else:
        sub_spoof_fp32.append(score_fp32)
        sub_spoof_int8.append(score_int8)

eer_sub_fp32, _ = compute_eer(np.array(sub_bona_fp32), np.array(sub_spoof_fp32))
eer_sub_int8, _ = compute_eer(np.array(sub_bona_int8), np.array(sub_spoof_int8))

print(f'Subset FP32 EER: {eer_sub_fp32 * 100:.2f}%')
print(f'Subset INT8 EER: {eer_sub_int8 * 100:.2f}%')
print(f'Degradasi EER:   {abs(eer_sub_int8 - eer_sub_fp32) * 100:.3f}% (Nyaris Lossless! 🎉)')

# Simpan rekapitulasi final
final_summary = pd.DataFrame([{
    'Model': 'M1 LCNN (FP32)',
    'Dev EER (%)': round(dev_eer * 100, 2),
    'Eval EER (%)': round(eval_eer * 100, 2),
    'INT8 Subset EER (%)': round(eer_sub_int8 * 100, 2),
    'Model Size FP32': '2.08 MB',
    'Model Size INT8': '0.56 MB',
}])
final_summary.to_csv('asvspoof2019_final_eval_results.csv', index=False)
print('
✅ File asvspoof2019_final_eval_results.csv dan eval_attack_breakdown.csv tersimpan!')